In [1]:
import cv2
import numpy as np
from pathlib import Path
from ultralytics.models.sam import SAM3SemanticPredictor
import torch

In [2]:
overrides = dict(
    conf=0.5,
    task="segment",
    mode="predict",
    model="sam3_model.pt",
    half=True,
    imgsz=644
)
predictor = SAM3SemanticPredictor(overrides=overrides)

In [3]:
data_dir = Path("newScans3")
supported_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

images_dir = data_dir/"images"
seg_output_dir = data_dir / "output_segmentation"
obb_output_dir = data_dir / "output_obb"
obb_labels_dir = data_dir / "labels"

seg_output_dir.mkdir(parents=True, exist_ok=True)
obb_output_dir.mkdir(parents=True, exist_ok=True)
obb_labels_dir.mkdir(parents=True, exist_ok=True)

In [4]:
print(f"Running SAM 3 inference on all images in folder: '{images_dir}'")
results = predictor(images_dir, text=["photo"],stream=True)
with torch.no_grad():
    for result in results:
        img_path = Path(result.path)
        if img_path.suffix.lower() not in supported_extensions:
            continue

        img_h, img_w = result.orig_shape
        obb_annotations = []
        raw_obb_boxes = []

        annotated_frame = result.plot(boxes=False, labels=True)
        seg_image_path = seg_output_dir / f"seg_{img_path.name}"
        cv2.imwrite(str(seg_image_path), annotated_frame)

        if result.masks is not None:
            for segments in result.masks.xy:
                if len(segments) == 0:
                    continue


                mask_canvas = np.zeros((img_h, img_w), dtype=np.uint8)
                polygon = np.array(segments, dtype=np.int32)
                cv2.fillPoly(mask_canvas, [polygon], 255)

                k=5
                kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (k, k))
                cleaned_canvas = cv2.morphologyEx(mask_canvas, cv2.MORPH_OPEN, kernel)

                sub_contours, _ = cv2.findContours(cleaned_canvas, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

                total_image_area = img_h * img_w
                noise_threshold = total_image_area * 0.05

                for sub_contour in sub_contours:
                    sub_area = cv2.contourArea(sub_contour)
                    if sub_area < noise_threshold:
                        continue

                # total_image_area = img_h * img_w
                # noise_threshold = total_image_area * 0.05
                # area = cv2.contourArea(polygon)
                # if area < noise_threshold:
                #     continue

                    rect = cv2.minAreaRect(sub_contour)
                    box_points = cv2.boxPoints(rect)
                    raw_obb_boxes.append(box_points.astype(np.int32))

                    normalized_points = []
                    for pt in box_points:
                        norm_x = pt[0] / img_w
                        norm_y = pt[1] / img_h
                        normalized_points.extend([norm_x, norm_y])

                    obb_annotations.append(normalized_points)

        txt_output_path = obb_labels_dir/ img_path.with_suffix(".txt").name
        class_idx = 0

        with open(txt_output_path, "w") as f:
            for ann in obb_annotations:
                coord_str = " ".join([f"{coord:.6f}" for coord in ann])
                f.write(f"{class_idx} {coord_str}\n")

        obb_visual_frame = result.orig_img.copy()
        for box in raw_obb_boxes:
            cv2.drawContours(obb_visual_frame, [box], 0, (0, 255, 0), 5)

        obb_image_path = obb_output_dir / f"obb_{img_path.name}"
        cv2.imwrite(str(obb_image_path), obb_visual_frame)

        print(f"Successfully Split and Processed: {img_path.name}")
        print(f"  -> Saved Mask Preview to: {seg_image_path}")
        print(f"  -> Saved OBB Matrix and Preview to: {obb_output_dir}\\")

        del result
        torch.cuda.empty_cache()

print("\nFinished split-folder exporting process!")

Running SAM 3 inference on all images in folder: 'newScans3\images'

Ultralytics 8.4.51  Python-3.13.12 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
image 1/197 C:\Ahmad\Programming\CropScans\newScans3\images\20250710123053_001.jpg: 644x644 3 photos, 1019.1ms
Successfully Split and Processed: 20250710123053_001.jpg
  -> Saved Mask Preview to: newScans3\output_segmentation\seg_20250710123053_001.jpg
  -> Saved OBB Matrix and Preview to: newScans3\output_obb\
image 2/197 C:\Ahmad\Programming\CropScans\newScans3\images\20250710123254_001.jpg: 644x644 2 photos, 189.2ms
Successfully Split and Processed: 20250710123254_001.jpg
  -> Saved Mask Preview to: newScans3\output_segmentation\seg_20250710123254_001.jpg
  -> Saved OBB Matrix and Preview to: newScans3\output_obb\
image 3/197 C:\Ahmad\Programming\CropScans\newScans3\images\20250710124312_001.jpg: 644x644 3 photos, 198.6ms
Successfully Split and Processed: 20250710124312_001.jpg
  -> Saved Mask Preview to: newS

In [5]:
from ultralytics.data.split import autosplit

In [6]:
autosplit(
    path=images_dir,
    weights=(0.8, 0.2, 0.0),
    annotated_only=True
)

Autosplitting images from newScans3\images, using *.txt labeled images only
: 100% ━━━━━━━━━━━━ 197/197 3.9Kit/s 0.1s


In [7]:
from ultralytics import YOLO

In [8]:
model = YOLO("yolo26n-obb.pt")

In [9]:

# Train the model
results = model.train(data="newScans3/dataset.yaml", epochs=100, imgsz=640,batch=-1,device=0,seed=0,
                      degrees=180.0,
workers=4,
    # flipud=0.5,
    # fliplr=0.5,
    # perspective=0.001,
    # shear=15.0
)

Ultralytics 8.4.51  Python-3.13.12 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=newScans3/dataset.yaml, degrees=180.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n-obb.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-33, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_

In [12]:
model.predict("unseen", save=True, imgsz=640, conf=0.25
              ,project="MyCropProject",name="unseen"
              )


WARNING OBB task does not support `save_crop`.
image 1/21 C:\Ahmad\Programming\CropScans\unseen\Scan_20260518.jpg: 640x480 2 photos, 24.8ms
WARNING OBB task does not support `save_crop`.
image 2/21 C:\Ahmad\Programming\CropScans\unseen\Scan_202605182 (10).jpg: 640x480 2 photos, 25.2ms
WARNING OBB task does not support `save_crop`.
image 3/21 C:\Ahmad\Programming\CropScans\unseen\Scan_202605182 (11).jpg: 640x480 3 photos, 25.1ms
WARNING OBB task does not support `save_crop`.
image 4/21 C:\Ahmad\Programming\CropScans\unseen\Scan_202605182 (12).jpg: 640x480 3 photos, 25.2ms
WARNING OBB task does not support `save_crop`.
image 5/21 C:\Ahmad\Programming\CropScans\unseen\Scan_202605182 (13).jpg: 640x480 3 photos, 25.1ms
WARNING OBB task does not support `save_crop`.
image 6/21 C:\Ahmad\Programming\CropScans\unseen\Scan_202605182 (14).jpg: 640x480 3 photos, 25.8ms
WARNING OBB task does not support `save_crop`.
image 7/21 C:\Ahmad\Programming\CropScans\unseen\Scan_202605182 (15).jpg: 640x480 


KeyboardInterrupt

